In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    DoubleType, StringType, TimestampType
)


In [2]:
from config import RAW_DIR, SPARK_PARQUET_DIR, SPARK_LOCAL_THREADS, SPARK_SHUFFLE_PARTITIONS

train_file = (RAW_DIR / "train.csv").as_posix()
metadata_file = (RAW_DIR / "building_metadata.csv").as_posix()
weather_file = (RAW_DIR / "weather_train.csv").as_posix()
output_dir = SPARK_PARQUET_DIR.as_posix()

spark = (
    SparkSession.builder
    .appName("ASHRAE_Electricity_Preprocessing")
    .master(f"local[{SPARK_LOCAL_THREADS}]")
    .config("spark.sql.shuffle.partitions", str(SPARK_SHUFFLE_PARTITIONS))
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
print("Spark version:", spark.version)


C:\Users\Fahad\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.3.0


In [3]:
# Use explicit schemas to avoid Spark scanning entire CSVs to infer data types.
train_schema = StructType([
    StructField("building_id", IntegerType(), True),
    StructField("meter", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("meter_reading", DoubleType(), True),
])

metadata_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("building_id", IntegerType(), True),
    StructField("primary_use", StringType(), True),
    StructField("square_feet", IntegerType(), True),
    StructField("year_built", DoubleType(), True),
    StructField("floor_count", DoubleType(), True),
])

weather_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("air_temperature", DoubleType(), True),
    StructField("cloud_coverage", DoubleType(), True),
    StructField("dew_temperature", DoubleType(), True),
    StructField("precip_depth_1_hr", DoubleType(), True),
    StructField("sea_level_pressure", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
])

train = spark.read.option("header", True).schema(train_schema).csv(train_file)
metadata = spark.read.option("header", True).schema(metadata_schema).csv(metadata_file)
weather = spark.read.option("header", True).schema(weather_schema).csv(weather_file)


In [4]:
# Check weather join keys: duplicates could create extra electricity rows.
weather_selected = weather.select(
    "site_id", "timestamp", "air_temperature", "dew_temperature", "wind_speed"
)
duplicate_weather_keys = (
    weather_selected.groupBy("site_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print("Duplicate weather (site_id, timestamp) keys:", duplicate_weather_keys)
if duplicate_weather_keys:
    raise ValueError("Weather join keys are not unique; investigate before joining.")


Duplicate weather (site_id, timestamp) keys: 0


In [5]:
# Match the existing DuckDB pipeline: meter 0 only, exclude Site 0,
# inner join building metadata, then left join weather on site and timestamp.
metadata_selected = metadata.select(
    "building_id", "site_id", "primary_use", "square_feet"
)

electricity = (
    train.filter(F.col("meter") == 0)
    .join(F.broadcast(metadata_selected), on="building_id", how="inner")
    .filter(F.col("site_id") != 0)
)

combined = electricity.join(
    weather_selected, on=["site_id", "timestamp"], how="left"
)

processed = (
    combined
    .withColumn("hour", F.hour("timestamp"))
    # Spark dayofweek: Sunday=1. DuckDB EXTRACT(DOW): Sunday=0.
    .withColumn("day_of_week", F.dayofweek("timestamp") - F.lit(1))
    .withColumn("month", F.month("timestamp"))
    .select(
        "building_id", "site_id", "primary_use", "square_feet",
        "timestamp", "hour", "day_of_week", "month",
        "air_temperature", "dew_temperature", "wind_speed",
        "meter_reading"
    )
)

# Spark writes a directory containing multiple Parquet part-files.
# This path is separate from the original DuckDB electricity_data.parquet file.
(
    processed.write
    .mode("overwrite")
    .option("compression", "zstd")
    .parquet(output_dir)
)
print("Processed Spark dataset saved to:", output_dir)


Processed Spark dataset saved to: D:/Big Data Project/data/processed/electricity_data_spark.parquet


In [6]:
# Read the saved Parquet output to verify the *actual written data*.
result = (
    spark.read.parquet(output_dir)
    .agg(
        F.count("*").alias("total_rows"),
        F.count("air_temperature").alias("matched_temperature"),
        F.round(
            F.lit(100.0) * F.count("air_temperature") / F.count("*"), 2
        ).alias("temperature_coverage_pct"),
        F.min("timestamp").alias("first_timestamp"),
        F.max("timestamp").alias("last_timestamp")
    )
)
result.show(truncate=False)

stats = result.first().asDict()
expected_rows = 11_152_501
expected_matched_temperature = 11_105_485
print("Row count matches DuckDB:", stats["total_rows"] == expected_rows)
print(
    "Matched-temperature count matches DuckDB:",
    stats["matched_temperature"] == expected_matched_temperature
)
if stats["total_rows"] != expected_rows or stats["matched_temperature"] != expected_matched_temperature:
    print("WARNING: Results differ from DuckDB; check schemas, joins and timestamps before continuing.")


+----------+-------------------+------------------------+-------------------+-------------------+
|total_rows|matched_temperature|temperature_coverage_pct|first_timestamp    |last_timestamp     |
+----------+-------------------+------------------------+-------------------+-------------------+
|11152501  |11105485           |99.58                   |2016-01-01 00:00:00|2016-12-31 23:00:00|
+----------+-------------------+------------------------+-------------------+-------------------+

Row count matches DuckDB: True
Matched-temperature count matches DuckDB: True


In [7]:
# Close the SparkSession after all preprocessing and checks are complete.
spark.stop()
print("SparkSession stopped.")


SparkSession stopped.
